***Dependencies***

In [1]:
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.4 MB/s eta 0:00:00


***Code***

In [6]:
import os
import glob
import zipfile
import shutil
import yaml
import urllib.request
from ultralytics import YOLO

# Define setup paths
dataset_dir = "/content/dataset"
target_zip = "/content/TheDataset.zip"
yaml_path = os.path.join(dataset_dir, "data.yaml")
train_txt = os.path.join(dataset_dir, "train.txt")
weights_path = "/content/yolov8n.pt"

# 1. AUTO-DETECT & RENAME ANY .ZIP TO 'TheDataset.zip'
found_zips = glob.glob("/content/*.zip")
other_zips = [f for f in found_zips if os.path.basename(f) != "TheDataset.zip"]

if other_zips:
    if os.path.exists(target_zip):
        os.remove(target_zip)
    os.rename(other_zips[0], target_zip)
    print(f"✅ Renamed '{os.path.basename(other_zips[0])}' ➔ 'TheDataset.zip'")

# 2. AUTO-UNZIP / REORGANIZE INTO /content/dataset
if not os.path.exists(yaml_path):
    print("🔍 Setting up dataset directory...")

    if os.path.exists("/content/data.yaml"):
        os.makedirs(dataset_dir, exist_ok=True)
        for item in ["data.yaml", "train.txt", "images", "labels"]:
            src = os.path.join("/content", item)
            dst = os.path.join(dataset_dir, item)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.move(src, dst)
        print("✅ Reorganized files into /content/dataset/")

    elif os.path.exists(target_zip):
        print("📦 Extracting 'TheDataset.zip' into /content/dataset/...")
        os.makedirs(dataset_dir, exist_ok=True)
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(dataset_dir)
        print("✅ Dataset successfully extracted!")
    else:
        raise FileNotFoundError("❌ Critical Error: No dataset zip found in /content/")
else:
    print("✅ Dataset directory active and ready!")

# 3. FIX train.txt PATHS
if os.path.exists(train_txt):
    with open(train_txt, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    fixed_lines = []
    for line in lines:
        if not line.startswith("/"):
            clean_path = line.lstrip("./")
            if clean_path.startswith("data/"):
                clean_path = clean_path[5:]
            fixed_path = os.path.join(dataset_dir, clean_path)
        else:
            fixed_path = line
        fixed_lines.append(fixed_path + "\n")

    with open(train_txt, "w") as f:
        f.writelines(fixed_lines)

    print(f"✅ Fixed {len(fixed_lines)} image paths in train.txt")

# 4. CONFIGURE data.yaml
with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

config["path"] = dataset_dir
config["train"] = "train.txt"
config["val"] = "train.txt"

with open(yaml_path, "w") as f:
    yaml.dump(config, f)

print("✅ data.yaml successfully configured!")

# 5. PRE-DOWNLOAD BASE WEIGHTS
if not os.path.exists(weights_path):
    print("⬇️ Pre-downloading base model weights (yolov8n.pt)...")
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
    urllib.request.urlretrieve(url, weights_path)
    print("✅ Base weights successfully downloaded!")
else:
    print("✅ yolov8n.pt already present!")

# 6. START MODEL TRAINING
model = YOLO(weights_path)

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="leaf_disease_run"
)

✅ Dataset directory active and ready!
✅ Fixed 450 image paths in train.txt
✅ data.yaml successfully configured!
✅ yolov8n.pt already present!
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

***Export***

In [9]:
import shutil
from google.colab import files

file = 3

# 1. Zip the train-4 directory
shutil.make_archive(f'train-{file}', 'zip', f'/content/runs/detect/leaf_disease_run/train-{file}')

# 2. Trigger browser download
files.download(f'train-{file}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

***Delete Training Data***

In [12]:
!rm -rf /content/runs/detect/leaf_disease_run/train-2